# Reconstruct pool state block-by-block

Turn the irregular swap events into one aligned, gap-free per-pool series
(price + active liquidity), trim to the study window, and save one CSV per pool
under `S.processed_dir`. Parameters come from `arblib.config.STUDY`.

In [1]:
%load_ext autoreload
%autoreload 2

from arblib import data_io, preprocessing as pp
from arblib import formulas as f
from arblib import modeling, regime
from arblib.config import STUDY as S, LIQUIDITY_FILES

REGIME = S.active_regime             # same window as extract (arblib.config)
if S.test_mode:
    win = regime.custom_window(S, S.test_start, S.test_end)
    print(f"[TEST MODE] processing custom window | study starts {win['study_start']} "
          f"(= test_start + {S.test_lead_min} min warm-up)")
else:
    win = regime.study_window(S, REGIME)
    print(f"processing regime: {REGIME} | study starts {win['study_start']}")
STUDY_START = win["study_start"]

[TEST MODE] processing custom window | study starts 2025-12-31 10:15:00 (= test_start + 15 min warm-up)


## 0. Load the raw swap extracts

In [2]:
dfs = data_io.load_pool_csvs(S.swaps_dir)

Loaded: df_uniswap_swap.csv
     amount0             amount1         dex  evt_block_number  \
0 -237871959   80000000000000000  uniswap_v3          24131870   
1   77424560  -26056274730796998  uniswap_v4          24131871   
2 -310555978  104444082993705986  uniswap_v3          24131871   
3   -7350000    2472651595603797  uniswap_v3          24131873   
4   82701586  -27808804678588136  uniswap_v3          24131873   

                evt_block_time  fee            liquidity  nb_swaps  \
0  2025-12-31 10:00:11.000 UTC  100   159135392615244153         1   
1  2025-12-31 10:00:23.000 UTC  500    17769293247749556         1   
2  2025-12-31 10:00:23.000 UTC  100   159135392615244153         3   
3  2025-12-31 10:00:59.000 UTC  500  1267918869826850600         1   
4  2025-12-31 10:00:59.000 UTC  100   159135392615244153         1   

                                                pool  \
0         0xe0554a476a092703abdb3ef35c80e0d76d32939f   
1  0x4f88f7c99022eace4740c6898f59ce6a2e798

## 1. End-of-block row per pool & block (already done in SQL)

The aggregated swap queries already return one end-of-block row per `(pool, block)` with
`nb_swaps` and `gas_price_max` / `gas_price_med`, so the old `count_swaps` + `clean_all`
(keep-latest-per-block) step is now a pass-through.

In [3]:
# nb_swaps and the single end-of-block row per (pool, block) are already produced by the
# aggregated Dune swap queries, so count_swaps / keep_latest are no longer needed here.
filtered_dfs = dfs

## 2. Split each DEX into one series per pool

In [4]:
pool_dfs = pp.split_by_pool(filtered_dfs)

Created uniswap_1: 210 rows
Created uniswap_2: 35 rows
Created uniswap_3: 93 rows
Created uniswap_4: 13 rows
Created uniswap_5: 3 rows
Created pancake_1: 73 rows
Created pancake_2: 32 rows

Total: 7 dataframes
['uniswap_1', 'uniswap_2', 'uniswap_3', 'uniswap_4', 'uniswap_5', 'pancake_1', 'pancake_2']


## 3. Drop pools that trade too rarely to reconstruct

In [5]:
pool_dfs, dropped_pools = pp.filter_pools_by_swap_gap(pool_dfs, S.max_gap_blocks)

k = 6000 blocks
Kept 7 pools

Kept pools:
  uniswap_1: 210 swaps, max consecutive gap = 6 blocks
  uniswap_2: 35 swaps, max consecutive gap = 40 blocks
  uniswap_3: 93 swaps, max consecutive gap = 18 blocks
  uniswap_4: 13 swaps, max consecutive gap = 59 blocks
  uniswap_5: 3 swaps, max consecutive gap = 204 blocks
  pancake_1: 73 swaps, max consecutive gap = 20 blocks
  pancake_2: 32 swaps, max consecutive gap = 49 blocks


## 3b. Drop dynamic-fee pools

The execution-price math needs a single fixed fee per pool, so pools whose `fee`
varies (e.g. Uniswap v4 dynamic-fee hooks) are excluded before saving.

In [6]:
pool_dfs, dropped_fee_pools = pp.filter_pools_by_constant_fee(pool_dfs)

Kept 7 constant-fee pools


## 4. Reconstruct a dense, forward-filled series per pool

In [7]:
reconstructed_pools, global_min, global_max, block_time_map = pp.reconstruct_pool_timeseries(pool_dfs)

Global block range: 24131870 to 24132167
Total blocks: 298

uniswap_1: 210 trades -> 298 blocks (100.0%)
uniswap_2: 35 trades -> 297 blocks (99.7%)
uniswap_3: 93 trades -> 295 blocks (99.0%)
uniswap_4: 13 trades -> 262 blocks (87.9%)
uniswap_5: 3 trades -> 228 blocks (76.5%)
pancake_1: 73 trades -> 295 blocks (99.0%)
pancake_2: 32 trades -> 288 blocks (96.6%)

Created 7 reconstructed time series


## 4b. Reconstruct active liquidity per block

Rebuild each pool's `liquidity` into the running active-liquidity state using the
mint/burn events (in-range deltas applied between swaps, held constant otherwise).

In [8]:
liq_dfs = data_io.load_pool_csvs(S.liquidity_dir, files=LIQUIDITY_FILES)
reconstructed_pools = pp.reconstruct_liquidity_states(reconstructed_pools, liq_dfs)

Loaded: df_uniswap_liq.csv
          dex  evt_block_number               evt_block_time  evt_index  \
0  uniswap_v3          24132147  2025-12-31 10:55:47.000 UTC         29   

   liquidity_delta                                        pool  tick_lower  \
0   80504858015188  0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640      196240   

   tick_upper  
0      196690  
Loaded: df_pancake_liq.csv
  event_type  evt_block_number               evt_block_time  evt_index  \
0       mint          24131870  2025-12-31 10:00:11.000 UTC        793   

    liquidity_delta                                        pool  tick_lower  \
0  1303875810537288  0x1445f32d1a74872ba41f3d8cf4022e9996120b31      194868   

   tick_upper  
0      197372  
uniswap_1: no mint/burn events, liquidity forward-filled
uniswap_2: no mint/burn events, liquidity forward-filled
uniswap_3 (0x88e6a0c2...): 1 mint/burn events, 1 applied in range
uniswap_4: no mint/burn events, liquidity forward-filled
uniswap_5: no mint/burn event

## 5a. MEV friction proxies

Two decay-weighted MEV series per pool: `mev_intensity` (recent top priority tip,
`gas_price_max - base_fee`) and `contest_frequency` (recent rate of same-block races,
`nb_swaps >= 2`), each decayed over past swap blocks with horizon `S.mev_horizon_blocks`.
No forward-fill — a quiet block inherits no stale competition value (gets `NaN`).

In [9]:
chain_gas = data_io.load_chain_gas(S.gas_path)
reconstructed_pools = f.add_mev_intensity(reconstructed_pools, chain_gas, S.mev_horizon_blocks)
reconstructed_pools = f.add_contest_freq(reconstructed_pools, S.mev_horizon_blocks)

uniswap_1: mev_intensity max 2.118e+09 wei
uniswap_2: mev_intensity max 1.981e+08 wei
uniswap_3: mev_intensity max 7.550e+08 wei
uniswap_4: mev_intensity max 3.213e+08 wei
uniswap_5: mev_intensity max 1.451e+07 wei
pancake_1: mev_intensity max 7.157e+08 wei
pancake_2: mev_intensity max 6.766e+08 wei
uniswap_1: nb_swaps_ewma max 1.88
uniswap_2: nb_swaps_ewma max 0.39
uniswap_3: nb_swaps_ewma max 0.79
uniswap_4: nb_swaps_ewma max 0.24
uniswap_5: nb_swaps_ewma max 0.12
pancake_1: nb_swaps_ewma max 0.67
pancake_2: nb_swaps_ewma max 0.40


## 5. Trim to the study window

In [10]:
filtered_pools = pp.filter_by_start_time(reconstructed_pools, STUDY_START)

uniswap_1: 298 -> 224 rows
uniswap_2: 298 -> 224 rows
uniswap_3: 298 -> 224 rows
uniswap_4: 298 -> 224 rows
uniswap_5: 298 -> 224 rows
pancake_1: 298 -> 224 rows
pancake_2: 298 -> 224 rows

Filtered all pools by time >= 2025-12-31 10:15:00


## 6. Save one CSV per pool

In [11]:
data_io.save_processed_pools(filtered_pools, S.processed_dir)

Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/data_analysis/processed/uniswap_1.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/data_analysis/processed/uniswap_2.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/data_analysis/processed/uniswap_3.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/data_analysis/processed/uniswap_4.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/data_analysis/processed/uniswap_5.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/data_analysis/processed/pancake_1.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/data_analysis/processed/pancake_2.csv
Done.


In [12]:
# quick sanity peek at one reconstructed pool
sample_pool = next(iter(filtered_pools))
filtered_pools[sample_pool][["nb_swaps", "gas_price_max", "mev_intensity", "nb_swaps_ewma"]].head(25)

,nb_swaps,gas_price_max,mev_intensity,nb_swaps_ewma
0,0,NaN,2.417808e+08,0.737402
1,1,5.963220e+07,2.262262e+08,0.754338
2,0,NaN,2.116362e+08,0.705688
3,2,1.147972e+08,2.012447e+08,0.789162
4,2,1.124457e+08,1.915234e+08,0.867253
5,0,NaN,1.791715e+08,0.811321
6,0,NaN,1.676161e+08,0.758997
7,0,NaN,1.568061e+08,0.710047
8,5,2.157372e+09,2.821285e+08,0.986719
9,3,6.504512e+07,2.640679e+08,1.116561


## 7. Save the common (pool-independent) modeling covariates

Persist the covariates every pool pair shares to `S.common_covariates_dir`
(`modeling/covariates/common_covariates/`):

- **`CEX_volatility.parquet`** — `[time, ewma_vol]`: RiskMetrics EWMA volatility of the
  `token0/token1` exchange rate `R = P_X/USD / P_Y/USD` (log returns differenced over time,
  `var_t = λ·var_{t-1} + (1-λ)·r_t²`, `λ = exp(-1/S.vol_horizon_min)`). Minute grid — joined to
  blocks by a backward merge on `time` downstream.
- **`chain_covariates.parquet`** — `[block_number, time, log_base_fee_per_gas, gas_util,
  log1p_tip_p50, log1p_tip_p90]`: base fee in logs, block fullness `gas_used/gas_limit`, and the
  per-block priority-tip p50/p90 as `log(1+tip)`. Joined by exact `block_number`.

Also creates (empty) `pool_pair_dependant_covariates/` for the per-pool-pair features built
later. Both files span the full extract window (warm-up included); downstream joins select the
study blocks.

In [13]:
x_usd = data_io.load_price_series(S.x_price_path)
y_usd = data_io.load_price_series(S.y_price_path)

modeling.save_common_covariates(x_usd, y_usd, chain_gas, S.common_covariates_dir, S.vol_horizon_min)
S.pair_covariates_dir.mkdir(parents=True, exist_ok=True)

Saved common covariates (CEX_volatility, chain_covariates) to /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/modeling/covariates/common_covariates


mev competition : max tip/block ->esssayer median tip ? ou autre quantiles 

-evenement multiple fenetre en fonctio de vol : petite/moy/big